# OPF PII Benchmarks — independent review

<!-- VOICE: 1–2 sentence hook in your voice. Suggested punchline:
"OpenAI shipped a PII filter. NVIDIA published numbers for their PII detector on three public benchmarks. Nobody compared them fairly. So I did." -->

This notebook is the technical artifact behind the post. It reproduces an independent review of [OpenAI's Privacy Filter (OPF)](https://github.com/openai/privacy-filter) against the three benchmarks NVIDIA published GLiNER-PII numbers on — Argilla PII, AI4Privacy, and Nemotron-PII — in three modes each (penalty / fair span / fair typed).

## TL;DR

<!-- VOICE: one sentence with the punchline + maybe the headline number. e.g. "On the strict-fair view, OPF lands at <X> vs NVIDIA's reported <Y> — but the per-category picture is much more interesting." -->

![Headline F1 by benchmark](results/figs/headline_f1.png)

In [1]:
# Render the headline chart from the full-size pre-computed run (results/).
# The image is generated by opf_benchmarks.charts (Phase 4 of the build plan) and
# committed to the repo. When viewing on GitHub/nbviewer this also renders the
# markdown image above. The code-cell fallback below is for Colab/local runs
# where the file may be missing.
from pathlib import Path
from IPython.display import Image, Markdown, display

hero = Path("results/figs/headline_f1.png")
if hero.exists():
    display(Image(str(hero)))
else:
    display(Markdown(
        "_Headline chart will appear here once `results/figs/headline_f1.png` "
        "is generated (see `opf_benchmarks/charts.py`)._"
    ))

_Headline chart will appear here once `results/figs/headline_f1.png` is generated (see `opf_benchmarks/charts.py`)._

## How this notebook works

The headline F1 chart above is **the real result**, computed against the full splits on an RTX 3090 and committed to `results/`. The cells below run the same pipeline **on a 100-example sample per benchmark** so a free-tier Colab session can finish in ~5 minutes. The sample run lands in `data_sample/` and `results_sample/`; it is for showing the pipeline works, not for the published numbers.

**Before running:**

1. Switch to a GPU runtime via *Runtime → Change runtime type → T4 GPU*. CPU works but takes longer.
2. Add a HuggingFace token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar). Two of the three datasets (`ai4privacy/pii-masking-300k` and `nvidia/Nemotron-PII`) are gated — accept their terms on HuggingFace first, then create a read token at https://huggingface.co/settings/tokens.

Then *Runtime → Run all*.

In [2]:
# --- Colab path: uncomment to clone the repo when running on Colab. ---
# Locally, skip this cell and use the next cell (which just verifies your cwd).

# import os
# REPO_DIR = "opf-benchmarks-geo"
# if not os.path.isdir(REPO_DIR):
#     !git clone https://github.com/CupOfGeo/opf-benchmarks-geo.git
# %cd {REPO_DIR}


In [3]:
# --- Local sanity check ---
# The notebook lives at the repo root, so cwd should already be correct in both
# Colab (after the clone+%cd above) and locally. If this assertion fails, you
# opened the notebook from somewhere unexpected.

import os
from pathlib import Path

assert Path("scripts").is_dir() and Path("opf_benchmarks").is_dir(), (
    f"cwd looks wrong: {os.getcwd()} — open this notebook from the repo root."
)
print(f"cwd: {os.getcwd()}")

cwd: /Users/georgemazzeo/Code/opf-benchmarks


In [4]:
# Install the package. On Colab this brings in the repo as `opf_benchmarks` +
# the OPF dependency. Locally, if you've already run `uv sync` (or `pip install -e .`),
# this is redundant and creates a stray `build/` directory — so we skip it.

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q .
else:
    print("Local mode — skipping `pip install .` (assumes the package is already installed).")

Local mode — skipping `pip install .` (assumes the package is already installed).


In [5]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except (ImportError, Exception):
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError(
        "HF_TOKEN not found. Add it as a Colab secret (key icon in the left sidebar, "
        "name it HF_TOKEN, toggle 'Notebook access' on). Two of the three datasets "
        "are gated — accept terms at https://huggingface.co/datasets/ai4privacy/pii-masking-300k "
        "and https://huggingface.co/datasets/nvidia/Nemotron-PII first, then create a "
        "read token at https://huggingface.co/settings/tokens."
    )

os.environ["HF_TOKEN"] = token
print("HF_TOKEN loaded.")

HF_TOKEN loaded.


In [6]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected — runs will be slow. Switch to T4 via Runtime → Change runtime type.")

Using device: cpu


## Knobs

This notebook runs on a **100-example sample per benchmark** so that a Colab session can finish in a few minutes. The full-size F1 numbers reported above (in the headline chart) come from a separate run against the full datasets on a GPU — the metrics for that run are committed to `results/` in the repo. The cells below write to `data/` and `results/` locally; when you clone fresh, they're empty, and your sample run populates them.

In [7]:
MAX_EXAMPLES = 100      # raise this to re-run a bigger sample; the published numbers used the full splits
BENCHMARKS = ["argilla", "ai4privacy", "nemotron"]

## Label mapping decisions

OPF was trained on **8 categories** (listed below). Each benchmark uses its own taxonomy — Argilla has ~55 labels, AI4Privacy ~60, Nemotron ~45 — and they overlap only partially with what OPF actually targets. So we hand-mapped each benchmark's labels to one of OPF's 8, or marked them out-of-scope.

The key decisions:

- **All financial / government identifiers collapse into `account_number`.** OPF doesn't distinguish SSN from credit card from IBAN from tax ID, so neither do we. Everything from `BITCOINADDRESS` to `medical_record_number` lands here.
- **Every flavor of name collapses into `private_person`** — first/last/middle/given/surname/title/prefix all map to the single category.
- **Demographic attributes are out of scope.** Age, gender, race, religion, political views, sexuality, education, occupation — none are PII in OPF's framing. Counting them against OPF would mis-frame the comparison.
- **Technical identifiers are out of scope.** IP addresses, MACs, user agents, device IDs, vehicle VINs, license plates — OPF treats these as network/asset metadata, not personal info.
- **Job titles, currency amounts, GPS coordinates, biometrics are out of scope** for the same reason — potentially identifying-in-aggregate, but not what OPF was trained to detect.

Out-of-scope spans are **deliberately kept in `_full.jsonl` with their original label** — that's what powers the "untyped × full" (penalty view) F1 in the report, and the per-source-label recall breakdown that shows exactly which categories OPF doesn't cover. The `_opfscope.jsonl` files drop them so typed evaluation is well-defined.

Full mapping below — see [`opf_benchmarks/label_map.py`](https://github.com/CupOfGeo/opf-benchmarks-geo/blob/main/opf_benchmarks/label_map.py) for the source.

In [8]:
from IPython.display import Markdown
from opf_benchmarks.label_map import MAPS, OPF_CATEGORIES

lines = ["### OPF's 8 target categories\n"]
for cat in sorted(OPF_CATEGORIES):
    lines.append(f"- `{cat}`")

lines.append("\n### Source labels grouped by OPF category\n")
for cat in sorted(OPF_CATEGORIES):
    lines.append(f"**`{cat}`**")
    for bench, table in MAPS.items():
        srcs = sorted(src for src, dst in table.items() if dst == cat)
        if srcs:
            lines.append(f"- *{bench}*: " + ", ".join(f"`{s}`" for s in srcs))
    lines.append("")

lines.append("### Out of OPF's scope\n")
lines.append(
    "These labels are kept in `_full.jsonl` with their original label (so the "
    "penalty-view F1 still counts them against OPF) and dropped from "
    "`_opfscope.jsonl` (so typed evaluation is well-defined):\n"
)
for bench, table in MAPS.items():
    oos = sorted(src for src, dst in table.items() if dst is None)
    lines.append(f"- *{bench}* ({len(oos)} labels): " + ", ".join(f"`{s}`" for s in oos))

Markdown("\n".join(lines))

### OPF's 8 target categories

- `account_number`
- `private_address`
- `private_date`
- `private_email`
- `private_person`
- `private_phone`
- `private_url`
- `secret`

### Source labels grouped by OPF category

**`account_number`**
- *argilla*: `ACCOUNTNUMBER`, `BIC`, `BITCOINADDRESS`, `CREDITCARDCVV`, `CREDITCARDISSUER`, `CREDITCARDNUMBER`, `ETHEREUMADDRESS`, `IBAN`, `LITECOINADDRESS`, `MASKEDNUMBER`, `SSN`
- *ai4privacy*: `ACCOUNTNUMBER`, `BIC`, `BITCOINADDRESS`, `CREDITCARD`, `CREDITCARDCVV`, `CREDITCARDNUMBER`, `DRIVERLICENSE`, `IBAN`, `IDCARD`, `PASSPORT`, `SOCIALNUMBER`, `TAXNUM`
- *nemotron*: `account_number`, `bank_routing_number`, `certificate_license_number`, `credit_card`, `customer_id`, `cvv`, `employee_id`, `health_plan_beneficiary_number`, `iban`, `medical_record_number`, `ssn`, `swift_bic`, `tax_id`

**`private_address`**
- *argilla*: `BUILDINGNUMBER`, `CITY`, `COUNTY`, `SECONDARYADDRESS`, `STATE`, `STREET`, `STREETADDRESS`, `ZIPCODE`
- *ai4privacy*: `BUILDING`, `BUILDINGNUMBER`, `CITY`, `COUNTRY`, `GEOCOORD`, `POSTCODE`, `SECADDRESS`, `STATE`, `STREET`, `ZIPCODE`
- *nemotron*: `address`, `city`, `country`, `county`, `postcode`, `state`, `street_address`

**`private_date`**
- *argilla*: `DATE`, `DOB`, `TIME`
- *ai4privacy*: `BOD`, `DATE`, `DATEOFBIRTH`, `TIME`
- *nemotron*: `date`, `date_of_birth`

**`private_email`**
- *argilla*: `EMAIL`
- *ai4privacy*: `EMAIL`
- *nemotron*: `email`

**`private_person`**
- *argilla*: `FIRSTNAME`, `LASTNAME`, `MIDDLENAME`, `PREFIX`
- *ai4privacy*: `FIRSTNAME`, `GIVENNAME`, `GIVENNAME1`, `GIVENNAME2`, `LASTNAME`, `LASTNAME1`, `LASTNAME2`, `LASTNAME3`, `MIDDLENAME`, `PREFIX`, `SURNAME`, `TITLE`
- *nemotron*: `first_name`, `full_name`, `last_name`, `middle_name`

**`private_phone`**
- *argilla*: `PHONE`, `PHONEIMEI`, `PHONENUMBER`, `PHONE_NUMBER`
- *ai4privacy*: `PHONE`, `PHONENUMBER`, `TEL`
- *nemotron*: `fax_number`, `phone_number`

**`private_url`**
- *argilla*: `URL`
- *ai4privacy*: `URL`
- *nemotron*: `url`

**`secret`**
- *argilla*: `PASSWORD`, `PIN`
- *ai4privacy*: `PASS`, `PASSWORD`, `PIN`
- *nemotron*: `api_key`, `password`, `pin`

### Out of OPF's scope

These labels are kept in `_full.jsonl` with their original label (so the penalty-view F1 still counts them against OPF) and dropped from `_opfscope.jsonl` (so typed evaluation is well-defined):

- *argilla* (26 labels): `ACCOUNTNAME`, `AGE`, `AMOUNT`, `COMPANYNAME`, `CURRENCY`, `CURRENCYCODE`, `CURRENCYNAME`, `CURRENCYSYMBOL`, `EYECOLOR`, `GENDER`, `HEIGHT`, `IP`, `IPV4`, `IPV6`, `JOBAREA`, `JOBTITLE`, `JOBTYPE`, `MAC`, `NEARBYGPSCOORDINATE`, `ORDINALDIRECTION`, `SEX`, `SEXTYPE`, `USERAGENT`, `USERNAME`, `VEHICLEVIN`, `VEHICLEVRM`
- *ai4privacy* (21 labels): `AGE`, `COMPANYNAME`, `ETHNICITY`, `EYECOLOR`, `GENDER`, `HEIGHT`, `IP`, `IPV4`, `IPV6`, `JOBAREA`, `JOBTITLE`, `JOBTYPE`, `MAC`, `NATIONALITY`, `POLITICAL`, `RELIGION`, `SEX`, `USERAGENT`, `USERNAME`, `VEHICLEVIN`, `VEHICLEVRM`
- *nemotron* (18 labels): `age`, `biometric_identifier`, `blood_type`, `coordinate`, `device_identifier`, `education_level`, `gender`, `ipv4`, `ipv6`, `license_plate`, `mac_address`, `occupation`, `political_view`, `race_ethnicity`, `religious_belief`, `sexuality`, `user_name`, `vehicle_identifier`

In [9]:
max_arg = f"--max-examples {MAX_EXAMPLES}" if MAX_EXAMPLES is not None else ""
!python -m scripts.download_datasets {max_arg} --benchmarks {" ".join(BENCHMARKS)}


=== argilla ===
  examples_full=100 examples_opfscope=89 spans_in=194 spans_out=120

=== ai4privacy ===
  examples_full=100 examples_opfscope=86 spans_in=499 spans_out=116

=== nemotron ===
[nemotron] WARNING: 8 label(s) not in label_map; treated as out-of-scope: {'company_name': 23, 'time': 19, 'credit_debit_card': 18, 'date_time': 15, 'employment_status': 9, 'http_cookie': 6, 'language': 5, 'unique_id': 2}
  examples_full=100 examples_opfscope=99 spans_in=475 spans_out=239


In [10]:
!python -m opf_benchmarks.run_eval --device {DEVICE} --benchmarks {" ".join(BENCHMARKS)}


=== [1/9] argilla/untyped_full ===
opf eval data/argilla_full.jsonl --device cpu --metrics-out results/argilla/untyped_full_metrics.json --eval-mode untyped --progress-every 10
[1/9] argilla/untyped_full: 100%|█████████████| 100/100 [01:14<00:00,  1.47ex/s]summary:
  field                        value                                                    
  ---------------------------  ---------------------------------------------------------
  examples                     100                                                      
  tokens                       4382                                                     
  windows                      100                                                      
  window_tokens                4382                                                     
  padded_window_tokens         4382                                                     
  elapsed_s                    73.6460                                                  
  tokens_per_s       

In [11]:
!python -m opf_benchmarks.aggregate

Wrote results/REPORT.md

--- Preview ---

# OpenAI Privacy Filter vs NVIDIA GLiNER-PII — Benchmark Results

All evals run on a 100-example sample of each benchmark using OPF's built-in `opf eval` (Apache 2.0, default checkpoint). Untyped mode ignores category identity (label-agnostic span detection). Typed mode scores the OPF-mapped category as well.

## Headline F1 (strict span match)

| Benchmark | NVIDIA GLiNER-PII (reported) | OPF — untyped × full | OPF — untyped × OPF-scope | OPF — typed × OPF-scope |
|---|---|---|---|---|
| Argilla PII | 0.70 | 0.674 | 0.629 | 0.531 |
| AI4Privacy | 0.64 | 0.821 | 0.765 | 0.756 |
| Nemotron-PII | 0.87 | 0.729 | 0.737 | 0.702 |

*untyped × full* = penalty view (every gold span counts, including categories OPF wasn't trained on). *untyped × OPF-scope* = label-agnostic F1 restricted to OPF-supported categories. *typed × OPF-scope* = strictest fair view, requires OPF predicts the right category too. The typed × OPF-scope number is the closest analogu

## Sample results

What the pipeline produces on the 100-example sample you just ran. The headline F1 numbers at the top of the notebook come from the full-size run in `results/`, not these.

In [12]:
from IPython.display import Markdown

Markdown(open("results/REPORT.md").read())

# OpenAI Privacy Filter vs NVIDIA GLiNER-PII — Benchmark Results

All evals run on a 100-example sample of each benchmark using OPF's built-in `opf eval` (Apache 2.0, default checkpoint). Untyped mode ignores category identity (label-agnostic span detection). Typed mode scores the OPF-mapped category as well.

## Headline F1 (strict span match)

| Benchmark | NVIDIA GLiNER-PII (reported) | OPF — untyped × full | OPF — untyped × OPF-scope | OPF — typed × OPF-scope |
|---|---|---|---|---|
| Argilla PII | 0.70 | 0.674 | 0.629 | 0.531 |
| AI4Privacy | 0.64 | 0.821 | 0.765 | 0.756 |
| Nemotron-PII | 0.87 | 0.729 | 0.737 | 0.702 |

*untyped × full* = penalty view (every gold span counts, including categories OPF wasn't trained on). *untyped × OPF-scope* = label-agnostic F1 restricted to OPF-supported categories. *typed × OPF-scope* = strictest fair view, requires OPF predicts the right category too. The typed × OPF-scope number is the closest analogue to NVIDIA's published strict F1.

## Token-level F1 (lenient)

Token-level F1 gives partial credit for any token overlap, even if span boundaries don't match exactly. Useful for sanity-checking the detector even when the boundary policy disagrees with the benchmark.

| Benchmark | untyped × full | untyped × OPF-scope | typed × OPF-scope |
|---|---|---|---|
| Argilla PII | 0.840 | 0.660 | 0.660 |
| AI4Privacy | 0.908 | 0.761 | 0.761 |
| Nemotron-PII | 0.815 | 0.728 | 0.728 |

## Per-source-label recall (untyped × full)

Shows what fraction of each gold label's character span is covered by *any* OPF prediction. Labels where OPF wasn't trained will naturally show low recall — this is the breakdown that explains the penalty-view F1.

### Argilla PII

| Source label | Recall | Recalled chars | Gold chars |
|---|---|---|---|
| `account_number` | 0.765 | 565 | 739 |
| `IPV6` | 1.000 | 351 | 351 |
| `private_address` | 0.377 | 129 | 342 |
| `private_person` | 0.807 | 246 | 305 |
| `private_phone` | 1.000 | 225 | 225 |
| `private_email` | 1.000 | 197 | 197 |
| `private_date` | 0.634 | 121 | 191 |
| `USERAGENT` | 0.000 | 0 | 148 |
| `private_url` | 0.000 | 0 | 140 |
| `IPV4` | 0.905 | 124 | 137 |
| `JOBTITLE` | 0.000 | 0 | 128 |
| `NEARBYGPSCOORDINATE` | 1.000 | 112 | 112 |
| `MAC` | 1.000 | 102 | 102 |
| `IP` | 1.000 | 91 | 91 |
| `USERNAME` | 0.867 | 78 | 90 |
| `secret` | 0.900 | 72 | 80 |
| `ACCOUNTNAME` | 0.000 | 0 | 76 |
| `AMOUNT` | 0.132 | 10 | 76 |
| `JOBAREA` | 0.000 | 0 | 72 |
| `VEHICLEVIN` | 1.000 | 68 | 68 |
| `GENDER` | 0.000 | 0 | 62 |
| `ORDINALDIRECTION` | 0.000 | 0 | 36 |
| `COMPANYNAME` | 0.412 | 14 | 34 |
| `HEIGHT` | 0.000 | 0 | 31 |
| `AGE` | 0.000 | 0 | 28 |
| `CURRENCY` | 0.458 | 11 | 24 |
| `JOBTYPE` | 0.000 | 0 | 23 |
| `EYECOLOR` | 0.000 | 0 | 21 |
| `CURRENCYNAME` | 0.000 | 0 | 15 |
| `CURRENCYSYMBOL` | 0.000 | 0 | 15 |
| `SEX` | 0.000 | 0 | 8 |
| `VEHICLEVRM` | 1.000 | 7 | 7 |
| `CURRENCYCODE` | 0.000 | 0 | 6 |

### AI4Privacy

| Source label | Recall | Recalled chars | Gold chars |
|---|---|---|---|
| `account_number` | 0.941 | 1552 | 1650 |
| `private_date` | 0.661 | 674 | 1019 |
| `private_address` | 0.832 | 657 | 790 |
| `IP` | 1.000 | 710 | 710 |
| `USERNAME` | 0.981 | 681 | 694 |
| `private_person` | 1.000 | 559 | 559 |
| `private_phone` | 0.986 | 428 | 434 |
| `private_email` | 1.000 | 302 | 302 |
| `SEX` | 0.000 | 0 | 175 |
| `secret` | 1.000 | 53 | 53 |

### Nemotron-PII

| Source label | Recall | Recalled chars | Gold chars |
|---|---|---|---|
| `private_url` | 0.234 | 296 | 1265 |
| `account_number` | 0.887 | 865 | 975 |
| `private_date` | 0.916 | 867 | 947 |
| `private_address` | 0.645 | 517 | 801 |
| `private_person` | 0.929 | 728 | 784 |
| `occupation` | 0.005 | 4 | 756 |
| `private_email` | 1.000 | 534 | 534 |
| `company_name` | 0.000 | 0 | 490 |
| `http_cookie` | 0.221 | 77 | 349 |
| `credit_debit_card` | 0.884 | 291 | 329 |
| `private_phone` | 0.714 | 180 | 252 |
| `date_time` | 1.000 | 217 | 217 |
| `biometric_identifier` | 0.852 | 144 | 169 |
| `time` | 0.117 | 15 | 128 |
| `secret` | 0.672 | 82 | 122 |
| `mac_address` | 0.950 | 113 | 119 |
| `religious_belief` | 0.000 | 0 | 119 |
| `coordinate` | 1.000 | 116 | 116 |
| `race_ethnicity` | 0.000 | 0 | 102 |
| `user_name` | 1.000 | 87 | 87 |
| `education_level` | 0.000 | 0 | 86 |
| `political_view` | 0.000 | 0 | 82 |
| `employment_status` | 0.000 | 0 | 79 |
| `unique_id` | 1.000 | 72 | 72 |
| `gender` | 0.000 | 0 | 44 |
| `sexuality` | 0.000 | 0 | 44 |
| `language` | 0.000 | 0 | 35 |
| `vehicle_identifier` | 1.000 | 33 | 33 |
| `ipv6` | 1.000 | 31 | 31 |
| `blood_type` | 0.000 | 0 | 29 |
| `age` | 0.000 | 0 | 14 |
| `ipv4` | 1.000 | 14 | 14 |
| `license_plate` | 1.000 | 7 | 7 |

## Per-OPF-category P/R/F1 (typed × OPF-scope)

### Argilla PII

| OPF category | Precision | Recall | F1 |
|---|---|---|---|
| `account_number` | 0.636 | 0.500 | 0.560 |
| `private_address` | 0.200 | 0.378 | 0.262 |
| `private_date` | 0.667 | 0.571 | 0.615 |
| `private_email` | 1.000 | 1.000 | 1.000 |
| `private_person` | 0.533 | 0.839 | 0.652 |
| `private_phone` | 0.636 | 0.538 | 0.583 |
| `private_url` | 0.000 | 0.000 | 0.000 |
| `secret` | 0.500 | 0.500 | 0.500 |

### AI4Privacy

| OPF category | Precision | Recall | F1 |
|---|---|---|---|
| `account_number` | 0.963 | 0.890 | 0.925 |
| `private_address` | 0.897 | 0.707 | 0.791 |
| `private_date` | 0.849 | 0.395 | 0.539 |
| `private_email` | 0.812 | 1.000 | 0.897 |
| `private_person` | 0.542 | 1.000 | 0.703 |
| `private_phone` | 0.853 | 0.935 | 0.892 |
| `private_url` | 0.000 | 0.000 | 0.000 |
| `secret` | 0.857 | 1.000 | 0.923 |

### Nemotron-PII

| OPF category | Precision | Recall | F1 |
|---|---|---|---|
| `account_number` | 0.639 | 0.790 | 0.707 |
| `private_address` | 0.424 | 0.537 | 0.474 |
| `private_date` | 0.777 | 0.913 | 0.839 |
| `private_email` | 1.000 | 1.000 | 1.000 |
| `private_person` | 0.516 | 0.927 | 0.663 |
| `private_phone` | 0.636 | 0.667 | 0.651 |
| `private_url` | 0.571 | 0.182 | 0.276 |
| `secret` | 0.571 | 0.286 | 0.381 |

## Sample sizes

| Benchmark | Examples (full) | Examples (OPF-scope) | Tokens (full) |
|---|---|---|---|
| Argilla PII | 100 | 89 | 4382 |
| AI4Privacy | 100 | 86 | 11382 |
| Nemotron-PII | 100 | 99 | 12978 |


## Bonus: run OPF on your own AI chats

<!-- VOICE: short intro to the personal-audit section. The post angle: "the benchmark is interesting; what catches your eye is what OPF flags in your own Claude Code / ChatGPT history." -->

A companion script reads your local Claude Code transcripts and runs OPF over them, then prints a category-by-category count of what it flagged. The script is in [`scripts/audit_claude_chats.py`](https://github.com/CupOfGeo/opf-benchmarks-geo/blob/main/scripts/audit_claude_chats.py) (added in a later phase — placeholder for now). Nothing personal leaves your machine; the script's default output is counts only.

## (Optional) Download the sample report

Run the cell below in Colab to download the sample `REPORT.md` to your machine. Skipped automatically when not running in Colab.

In [13]:
try:
    from google.colab import files
    files.download("results/REPORT.md")
except ImportError:
    print("Not running in Colab — skipping download.")

Not running in Colab — skipping download.
